### **Deportes**

Trabajo realizado por Alberto Sainz, José Emparanza e Inés Devis (Grupo 8)



*    **Fase 1: Dataset**

Recopilación automática del dataset (wikipedia API)

In [4]:
import re
import time
import requests
import pandas as pd

# 0. Parámetros de ejecución
# Si GENERAR_DATASET = False no se llama a Wikipedia y se descarga el CSV ya generado de nuestro GitHub
GENERAR_DATASET = True
URL_DATASET = "https://github.com/albertoosg/Practica_Mineria_de_Textos"  # TODO: poner la URL de nuestro repo
MIN_PARRAFOS_CLASE = 350   # si una clase no llega a este mínimo se completa con artículos de su categoría de Wikipedia
MAX_PARRAFOS_CLASE = 450   # tope por clase para que el dataset quede equilibrado

# 1. Artículos de Wikipedia en Esperanto organizados por categorías
SEEDS = {
    "futbol": [
        "Futbalo", "Historio de futbalo", "Futbala Mondpokalo", "UEFA", "FIFA",
        "UEFA-ĉampiona Ligo", "FC Barcelona", "Real Madrid", "Pelé", "Diego Maradona",
        "Lionel Messi", "Cristiano Ronaldo", "Brazila nacia teamo de futbalo",
        "Germana nacia teamo de futbalo", "Arĝentina nacia teamo de futbalo",
        "Itala nacia teamo de futbalo", "Franca nacia teamo de futbalo", "Hispana nacia teamo de futbalo",
        "Angla nacia teamo de futbalo", "Urugvaja nacia teamo de futbalo", "Nederlanda nacia teamo de futbalo",
        "Golejo", "Pruvo de ofsajdo", "Eŭropa Futbal-Ĉampionado",
        "Zinedine Zidane", "Johan Cruijff", "Marta (futbalistino)", "Futbalisto",
        "Flava karto", "Ruĝa karto", "Futbala Mondpokalo 2022", "Futbala Mondpokalo 2018",
        "Futbala Mondpokalo 2014", "Futbala Mondpokalo 2010", "Futbala Mondpokalo 2006",
        "Copa América", "Premier League", "La Liga", "Serie A (Italio)", "Bundesliga"
    ],
    "baloncesto": [
        "Basketbalo", "Korbopilko", "NBA", "FIBA", "FIBA Korbopilka Mondpokalo", "EuroLeague",
        "Michael Jordan", "Kobe Bryant", "LeBron James", "Shaquille O'Neal",
        "Magic Johnson", "Larry Bird", "Pau Gasol", "Marc Gasol",
        "Usona vira nacia teamo de basketbalo", "Hispana nacia teamo de basketbalo",
        "Litova nacia teamo de basketbalo", "Argentina nacia teamo de basketbalo",
        "Greka nacia teamo de basketbalo", "Korbopilka tribunalo", "Tri-punkta pafo",
        "Danko (korbopilko)", "Stephen Curry", "Luka Dončić", "Dirk Nowitzki", "WNBA",
        "Boston Celtics", "Chicago Bulls", "Los Angeles Lakers", "Golden State Warriors",
        "San Antonio Spurs", "Miami Heat", "New York Knicks", "Kareem Abdul-Jabbar",
        "Wilt Chamberlain", "Bill Russell", "Tim Duncan", "Hakeem Olajuwon",
        "Giannis Antetokounmpo", "Nikola Jokić", "EuroBasket"
    ],
    "tenis": [
        "Teniso", "Historio de teniso", "Granda Ŝlemo (teniso)", "Wimbledon (teniso)",
        "Roland Garros", "US Open (teniso)", "Australian Open (teniso)",
        "Rafael Nadal", "Roger Federer", "Novak Djoković", "Serena Williams",
        "Carlos Alcaraz", "Davis-Pokalo", "Tenisejo", "Tenisa pilko", "Tenisa rakedo",
        "Steffi Graf", "Martina Navratilova", "Björn Borg", "ATP Tour", "Virina Tenisa Asocio",
        "Servobato (teniso)", "Andy Murray", "Venus Williams",
        "Arantxa Sánchez Vicario", "Boris Becker", "Andre Agassi", "Pete Sampras",
        "Rod Laver", "John McEnroe", "Jimmy Connors", "Ivan Lendl", "Mats Wilander",
        "Stefan Edberg", "Monica Seles", "Chris Evert", "Billie Jean King",
        "Fed Cup", "ATP Finals", "Opcio (teniso)"
    ]
}

# Categorías de Wikipedia que usamos para completar las clases que se quedan cortas
CATEGORIAS = {
    "futbol": "Kategorio:Futbalo",
    "baloncesto": "Kategorio:Basketbalo",
    "tenis": "Kategorio:Teniso"
}

URL_API = "https://eo.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "MineriaTextosBot/1.0 (contacto: estudiante@cunef.edu) Python-requests"
}

# 2. Conexión segura con la API de Wikipedia en Esperanto
def obtener_texto_wikipedia(titulo: str) -> str:
    params = {
        "action": "query",
        "format": "json",
        "titles": titulo,
        "prop": "extracts",
        "explaintext": True,
        "redirects": 1
    }

    # Intentos múltiples para evitar fallos de red puntuales
    for _ in range(3):
        try:
            r = requests.get(URL_API, params=params, headers=HEADERS, timeout=12)
            if r.status_code == 200:
                data = r.json()
                pages = data.get("query", {}).get("pages", {})
                for _, page_info in pages.items():
                    if "extract" in page_info:
                        return page_info["extract"]
                return ""
            elif r.status_code == 429:
                time.sleep(2)
        except Exception:
            time.sleep(1)
    return ""

# Devuelve los títulos de los artículos de una categoría (y de sus subcategorías, un nivel)
def obtener_articulos_categoria(categoria: str, profundidad: int = 1) -> list:
    params = {
        "action": "query",
        "format": "json",
        "list": "categorymembers",
        "cmtitle": categoria,
        "cmtype": "page|subcat",
        "cmlimit": 500
    }
    try:
        r = requests.get(URL_API, params=params, headers=HEADERS, timeout=12)
        miembros = r.json().get("query", {}).get("categorymembers", [])
    except Exception:
        return []

    titulos = []
    for m in miembros:
        if m["ns"] == 0:  # ns 0 = artículo
            titulos.append(m["title"])
        elif m["ns"] == 14 and profundidad > 0:  # ns 14 = subcategoría
            titulos.extend(obtener_articulos_categoria(m["title"], profundidad - 1))
            time.sleep(0.35)
    return titulos

# 3. Limpieza de datos y filtrado de ruido
def limpiar_y_extraer_parrafos(texto_articulo: str, min_longitud: int = 60) -> list:
    parrafos_crudos = texto_articulo.split("\n")
    parrafos_limpios = []

    for p in parrafos_crudos:
        p = p.strip()
        # Omitir títulos o secciones de Wikipedia
        if p.startswith("=") and p.endswith("="):
            continue

        # Reemplazar el punto y coma por coma para respetar el contrato del CSV (text;class)
        p = p.replace(";", ",")
        p = re.sub(r"\s+", " ", p)

        # Filtrar por longitud mínima y descartar párrafos con exceso de números (ej. listas de años)
        if len(p) >= min_longitud:
            num_digitos = sum(c.isdigit() for c in p)
            if (num_digitos / len(p)) < 0.25:
                parrafos_limpios.append(p)

    return parrafos_limpios

# 4. Proceso principal de extracción y guardado en CSV
nombre_archivo = "dataset_esperanto_deportes2.csv"

if GENERAR_DATASET:
    registros = []
    titulos_usados = set()  # para no usar el mismo artículo en dos clases

    for categoria, articulos in SEEDS.items():
        print(f"Extrayendo: {categoria.upper()}...")
        parrafos_cat = []
        textos_cat = set()
        no_encontrados = []

        for art in articulos:
            txt = obtener_texto_wikipedia(art)
            if not txt:
                no_encontrados.append(art)
                continue
            titulos_usados.add(art)
            for p in limpiar_y_extraer_parrafos(txt):
                if p not in textos_cat:
                    textos_cat.add(p)
                    parrafos_cat.append({"text": p, "class": categoria})
            time.sleep(0.35)

        print(f" -> Semillas: {len(parrafos_cat)} párrafos")
        if no_encontrados:
            print(f" -> Artículos no encontrados ({len(no_encontrados)}): {no_encontrados}")

        # Si la clase no llega al mínimo, completamos con artículos de su categoría
        if len(parrafos_cat) < MIN_PARRAFOS_CLASE:
            extra = obtener_articulos_categoria(CATEGORIAS[categoria])
            añadidos = 0
            for art in extra:
                if len(parrafos_cat) >= MIN_PARRAFOS_CLASE:
                    break
                if art in titulos_usados:
                    continue
                titulos_usados.add(art)
                txt = obtener_texto_wikipedia(art)
                for p in limpiar_y_extraer_parrafos(txt):
                    if p not in textos_cat:
                        textos_cat.add(p)
                        parrafos_cat.append({"text": p, "class": categoria})
                añadidos += 1
                time.sleep(0.35)
            print(f" -> Completado con {añadidos} artículos de {CATEGORIAS[categoria]}")

        df_temp = pd.DataFrame(parrafos_cat)
        # Recortar las clases muy grandes para equilibrar (con semilla fija para que sea reproducible)
        if len(df_temp) > MAX_PARRAFOS_CLASE:
            df_temp = df_temp.sample(n=MAX_PARRAFOS_CLASE, random_state=42)
        registros.extend(df_temp.to_dict(orient="records"))
        print(f" -> {categoria}: {len(df_temp)} párrafos")

    # Crear el DataFrame final, mezclarlo aleatoriamente y guardarlo cumpliendo el contrato de datos
    df_dataset = pd.DataFrame(registros)
    # Si un mismo párrafo aparece en dos clases lo quitamos de las dos
    df_dataset = df_dataset.drop_duplicates(subset=["text"], keep=False)
    df_dataset = df_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    df_dataset.to_csv(nombre_archivo, sep=";", index=False, encoding="utf-8")
else:
    print("Cargando el dataset ya generado desde GitHub...")
    df_dataset = pd.read_csv(URL_DATASET, sep=";", encoding="utf-8")

print("\n--- RESUMEN FINAL DEL DATASET---")
print(df_dataset["class"].value_counts())
print(f"Total párrafos: {len(df_dataset)}")

Extrayendo: FUTBOL...
 -> Semillas: 230 párrafos
 -> Artículos no encontrados (6): ['Historio de futbalo', 'UEFA-ĉampiona Ligo', 'Arĝentina nacia teamo de futbalo', 'Pruvo de ofsajdo', 'Flava karto', 'Serie A (Italio)']
 -> Completado con 17 artículos de Kategorio:Futbalo
 -> futbol: 382 párrafos
Extrayendo: BALONCESTO...
 -> Semillas: 143 párrafos
 -> Artículos no encontrados (18): ['FIBA Korbopilka Mondpokalo', 'EuroLeague', 'Marc Gasol', 'Usona vira nacia teamo de basketbalo', 'Hispana nacia teamo de basketbalo', 'Litova nacia teamo de basketbalo', 'Argentina nacia teamo de basketbalo', 'Greka nacia teamo de basketbalo', 'Korbopilka tribunalo', 'Tri-punkta pafo', 'Danko (korbopilko)', 'Dirk Nowitzki', 'Wilt Chamberlain', 'Bill Russell', 'Tim Duncan', 'Hakeem Olajuwon', 'Giannis Antetokounmpo', 'Nikola Jokić']
 -> Completado con 0 artículos de Kategorio:Basketbalo
 -> baloncesto: 143 párrafos
Extrayendo: TENIS...
 -> Semillas: 171 párrafos
 -> Artículos no encontrados (9): ['Historio

Medición del ruido del dataset (párrafos genéricos que heredan la etiqueta del artículo)

In [5]:
# Medición aproximada del ruido: un párrafo se considera "genérico" si no contiene
# ninguna palabra clave de su deporte (buscamos raíces, así cubrimos plurales, acusativos, etc.)
PALABRAS_CLAVE = {
    "futbol": ["futbal", "golo", "golej", "golist", "ofsajd", "penalt", "fifa", "uefa", "mondpokal"],
    "baloncesto": ["basketbal", "korbopilk", "korbo", "nba", "fiba", "wnba", "euroleague", "eurobasket"],
    "tenis": ["tenis", "raked", "ŝlem", "wimbledon", "garros", "atp", "wta", "davis"]
}

def tiene_palabra_clave(texto: str, clase: str) -> bool:
    texto = texto.lower()
    return any(raiz in texto for raiz in PALABRAS_CLAVE[clase])

df_dataset["generico"] = [not tiene_palabra_clave(t, c) for t, c in zip(df_dataset["text"], df_dataset["class"])]

print("Porcentaje de párrafos sin ninguna palabra clave de su clase (ruido estimado):")
print((df_dataset.groupby("class")["generico"].mean() * 100).round(1).astype(str) + " %")

print("\nAlgunos ejemplos de párrafos genéricos:")
for clase in SEEDS:
    ejemplos = df_dataset[(df_dataset["class"] == clase) & (df_dataset["generico"])].head(2)
    for t in ejemplos["text"]:
        print(f" [{clase}] {t[:150]}...")

# Quitamos la columna auxiliar para que el dataset siga cumpliendo el contrato text;class
df_dataset = df_dataset.drop(columns=["generico"])

Porcentaje de párrafos sin ninguna palabra clave de su clase (ruido estimado):
class
baloncesto    36.4 %
futbol        29.1 %
tenis         26.9 %
Name: generico, dtype: object

Algunos ejemplos de párrafos genéricos:
 [futbol] interago en la ludado - se li tuŝas aŭ ludas la pilkon post pasigo aŭ tuŝo de samteamano....
 [futbol] 2001 San Jose Earthquakes 2-1 p.d. Los Angeles Galaxy (raporto anglalingva, arkivigite)...
 [baloncesto] Laŭ raporto de la Instituto por Diverseco kaj Etiko en Sporto ĉe la Universitato de Centra Florido, en la sezono de 2023, 63.8% de la ludantoj de la l...
 [baloncesto] pafgardisto: Tiu ludanto – ĵetanto - helpas organizanton ĉe konstruo de la ludo kaj akiras poentojn el distanco....
 [tenis] Ortangula ludkampo apartiĝas per reto en du duonojn. Tuta ludkampo 78 futojn (23,77 metrojn) longas (ĉiu flanko, do 39 futojn = 11,89 metrojn longa). ...
 [tenis] Neniu ludanto rajtas tuŝi la reton, tio estus malvenko de la poento....




*   **Fase 2: NLP**







Modelo NLP para Esperanto (estilo spacy, sin el uso de librerías externas)

In [6]:
import re
import requests
from typing import List, Optional

class Token:
    """Representa un token individual con sus atributos morfológicos y sintácticos."""
    def __init__(self, text: str, lemma: str, pos: str, is_stop: bool, is_punct: bool, i: int = 0):
        self.text = text
        self.lemma = lemma
        self.pos = pos
        self.is_stop = is_stop
        self.is_punct = is_punct
        self.i = i  # posición del token dentro del Doc (como en spaCy)

    # Atributos con guion bajo para que el API se parezca más al de spaCy
    @property
    def lemma_(self):
        return self.lemma

    @property
    def pos_(self):
        return self.pos

    @property
    def lower_(self):
        return self.text.lower()

    @property
    def is_alpha(self):
        return self.text.isalpha()

    def __repr__(self):
        return f"Token({self.text} -> lemma='{self.lemma}', pos='{self.pos}', stop={self.is_stop})"


class Doc:
    """Secuencia estructurada de tokens iterable y con acceso por índice."""
    def __init__(self, text: str, tokens: List[Token]):
        self.text = text
        self.tokens = tokens

    def __iter__(self):
        return iter(self.tokens)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, index):
        return self.tokens[index]

    def __repr__(self):
        return f"Doc(length={len(self.tokens)})"


class EsperantoNLP:
    """
    Pipeline de NLP para Esperanto basado en reglas gramaticales y morfológicas:
    - Sustantivos: terminan en -o (plurales -oj, acusativos -on, -ojn)
    - Adjetivos: terminan en -a (plurales -aj, acusativos -an, -ajn)
    - Adverbios derivados: terminan en -e
    - Verbos: -i, -as, -is, -os, -us, -u
    """

    # Construcción concisa de correlativos, pronombres, preposiciones y conjunciones
    CORRELATIVES = {
        p + s
        for p in ["ti", "ki", "i", "ĉi", "neni"]
        for s in ["u", "un", "uj", "ujn", "o", "on", "a", "an", "aj", "ajn", "e", "en", "el", "al", "om", "am", "es"]
    }

    PRONOUNS = set("mi min vi vin li lin ŝi ŝin ĝi ĝin ni nin ili ilin oni onin si sin".split())
    PREPOSITIONS = set("al anstataŭ antaŭ apud ĉe ĉirkaŭ da de dum ekster el en ĝis inter je kontraŭ krom kun laŭ malgraŭ per po por post preter pri pro sen sub super sur tra trans".split())
    CONJUNCTIONS = set("kaj aŭ sed ĉar se ke kvankam nek do tamen".split())

    # Palabras que no siguen las terminaciones regulares y que hay que reconocer antes de aplicar las reglas
    # (si no, por ejemplo "tri" o "plu" acaban como verbos por terminar en -i / -u)
    NUMERALS = set("nul unu du tri kvar kvin ses sep ok naŭ dek cent mil".split())
    PARTICLES = set("ne ĉu jes jen ajn ja ĉi nu".split())
    ADVERBS = set("ankaŭ ankoraŭ apenaŭ baldaŭ hieraŭ hodiaŭ morgaŭ nun jam ĵus tuj tre tro plu pli plej nur eĉ for preskaŭ kvazaŭ mem ju des".split())

    def __init__(self, custom_stop_words: Optional[set] = None):
        self.stop_words = custom_stop_words if custom_stop_words is not None else self._cargar_stop_words()

    def _cargar_stop_words(self) -> set:
        url = "https://raw.githubusercontent.com/jparisu/nlp-esperantilo/main/resources/esperanto/vortoj.json"
        try:
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                data = r.json()
                # El JSON no es una lista de palabras: las palabras están dentro de la clave "vortoj",
                # cada una como un diccionario con "vorto" (la palabra) e "ignorinda" (si es stop-word)
                words = {e["vorto"] for e in data["vortoj"] if e.get("ignorinda", True)}
                print(f"[NLP] Stop-words cargadas ({len(words)} palabras).")
                return words
        except Exception:
            pass

        print("[NLP] Usando conjunto base de stop-words.")
        base = set("la estas esti ne ĉi ankaŭ kiel pli ol tre jam nur".split())
        return base | self.PRONOUNS | self.PREPOSITIONS | self.CONJUNCTIONS

    def _tokenizar(self, text: str) -> List[str]:
        patron = r"[\wĉĝĥĵŝŭĈĜĤĴŜŬ]+|[^\w\s]"
        return re.findall(patron, text)

    # Devuelve siempre (pos, lema, is_punct)
    def _analizar_morfologia(self, token_str: str, inicio_frase: bool = False):
        if not re.search(r"[\wĉĝĥĵŝŭĈĜĤĴŜŬ]", token_str):
            return "PUNCT", token_str, True

        w = token_str.lower()

        if w in self.PRONOUNS:
            # Los acusativos (min, lin, ilin...) se lematizan quitando la -n
            lemma = w[:-1] if w.endswith("n") else w
            return "PRON", lemma, False

        if w in self.CORRELATIVES:
            # Igual con los correlativos en acusativo (kiun -> kiu, tion -> tio)
            lemma = w[:-1] if w.endswith("n") and w[:-1] in self.CORRELATIVES else w
            return "PRON", lemma, False

        if w in self.PREPOSITIONS:
            return "ADP", w, False

        if w in self.CONJUNCTIONS:
            return "CCONJ", w, False

        if re.fullmatch(r"\d+", w) or w in self.NUMERALS:
            return "NUM", w, False

        if w == "la":
            return "DET", "la", False

        if w in self.PARTICLES:
            return "PART", w, False

        if w in self.ADVERBS:
            return "ADV", w, False

        # Nombres propios: empiezan por mayúscula y no están al principio de la frase
        # (si están al principio de la frase no podemos distinguirlos y se aplican las reglas normales)
        if token_str[0].isupper() and not inicio_frase:
            return "PROPN", token_str, False

        # Desinencias verbales
        """terminación que se añade a la raíz de un verbo
        (el lexema que aporta el significado principal)
        para indicar sus accidentes gramaticales, tales
        como el tiempo (presente, pasado, futuro), el modo
        (indicativo, subjuntivo, condicional, imperativo),
        la persona y el número. """

        for ending in ("as", "is", "os", "us", "u", "i"):
            if w.endswith(ending) and len(w) > len(ending) + 1:
                return "VERB", w[:-len(ending)] + "i", False

        # Sustantivos
        for ending in ("ojn", "oj", "on", "o"):
            if w.endswith(ending) and len(w) > len(ending) + 1:
                return "NOUN", w[:-len(ending)] + "o", False

        # Adjetivos
        for ending in ("ajn", "aj", "an", "a"):
            if w.endswith(ending) and len(w) > len(ending) + 1:
                return "ADJ", w[:-len(ending)] + "a", False

        # Adverbios derivados
        if w.endswith("e") and len(w) > 2:
            return "ADV", w[:-1] + "e", False

        return "X", w, False

    def __call__(self, text: str) -> Doc:
        tokens_obj = []
        inicio_frase = True
        for i, raw_tok in enumerate(self._tokenizar(text)):
            pos, lemma, is_punct = self._analizar_morfologia(raw_tok, inicio_frase)
            is_stop = (raw_tok.lower() in self.stop_words) or (lemma.lower() in self.stop_words)
            tokens_obj.append(Token(
                text=raw_tok,
                lemma=lemma,
                pos=pos,
                is_stop=is_stop,
                is_punct=is_punct,
                i=i
            ))
            # El siguiente token es inicio de frase si este es un punto, ? o !
            if raw_tok in ".!?":
                inicio_frase = True
            elif not is_punct:
                inicio_frase = False
        return Doc(text=text, tokens=tokens_obj)



# PRUEBA DE EJECUCIÓN DEL MODELO NLP

nlp = EsperantoNLP()
frases = [
    "La famaj futbalistoj rapide ludis en la granda hispana stadiono.",
    "Ĉu vi ŝatas tenison? Mi tre ŝatas ĝin, kaj ankaŭ mia frato.",
    "En 2022 Lionel Messi gajnis la Mondpokalon kontraŭ Francio.",
    "La du teamoj ludis plu, kaj tri ludantoj ricevis flavan karton.",
]

for frase in frases:
    doc = nlp(frase)
    print(f"\n>>> {frase}")
    print(f"{'Texto':<15} {'Lema':<15} {'POS':<8} {'Stop-Word':<10} {'Puntuación':<10}")
    print("-" * 62)
    for t in doc:
        print(f"{t.text:<15} {t.lemma:<15} {t.pos:<8} {str(t.is_stop):<10} {str(t.is_punct):<10}")

[NLP] Stop-words cargadas (122 palabras).

>>> La famaj futbalistoj rapide ludis en la granda hispana stadiono.
Texto           Lema            POS      Stop-Word  Puntuación
--------------------------------------------------------------
La              la              DET      True       False     
famaj           fama            ADJ      False      False     
futbalistoj     futbalisto      NOUN     False      False     
rapide          rapide          ADV      False      False     
ludis           ludi            VERB     False      False     
en              en              ADP      True       False     
la              la              DET      True       False     
granda          granda          ADJ      False      False     
hispana         hispana         ADJ      False      False     
stadiono        stadiono        NOUN     False      False     
.               .               PUNCT    False      True      

>>> Ĉu vi ŝatas tenison? Mi tre ŝatas ĝin, kaj ankaŭ mia frato.
Text